# 004 · The warnings Module

> **Standard Library › Logging & Config** · Python 3.14

Warnings tell the *developer* (not the end user) that something works today but is questionable: a deprecated API, a suspicious argument, a lossy conversion. Unlike exceptions they don't stop the program, and unlike log messages they are **filterable by category and location** and are shown only once per call site by default.

**In this notebook you will learn:**

- `warnings.warn` and how it relates to `[Obsolete]` and compiler warnings in C#
- the built-in warning categories and how to define your own
- `stacklevel` — pointing the warning at the caller
- the filter system: `simplefilter`, `filterwarnings`, and `"error"` to turn warnings into exceptions
- testing warnings with `catch_warnings(record=True)`
- `-W` / `PYTHONWARNINGS`, why `DeprecationWarning` is often invisible, `@warnings.deprecated` (3.13+) and `logging.captureWarnings`

## .NET mental model

| C# / .NET | Python |
|---|---|
| `[Obsolete("Use NewApi")]` — **compile-time** warning CS0618 | `@warnings.deprecated("Use new_api")` (3.13+) — **runtime** warning (type checkers also flag it statically) |
| `[Obsolete("...", error: true)]` | `-W error::DeprecationWarning` or `simplefilter("error")` |
| `#pragma warning disable CS0618` | `with warnings.catch_warnings(): warnings.simplefilter("ignore", DeprecationWarning)` |
| `<TreatWarningsAsErrors>true</TreatWarningsAsErrors>` | `python -W error` / `PYTHONWARNINGS=error` |
| `<NoWarn>CS0618</NoWarn>` | `filterwarnings("ignore", category=...)` |
| `Trace.TraceWarning(...)` / `ILogger.LogWarning` | `logging` — use that for operational problems |

The key difference: C# warnings are produced by the **compiler**; Python's are raised by **running code**. A deprecated function only warns when it's actually called.

**When to use which:** `warnings.warn` for problems the *calling code* should fix (wrong usage, deprecated API); `logging.warning` for runtime conditions an *operator* should know about (disk full, retrying a request); exceptions when you can't continue.

## Issuing a warning

`warnings.warn(message, category=UserWarning, stacklevel=1)`. Execution continues normally after the call.

In a notebook, warnings are printed to **stderr** (often shown with a red background) in the format `file:line: Category: message`. The file name is the temporary file Jupyter uses for the cell, so it differs per machine.

In [ ]:
import warnings

def set_volume(level: int) -> int:
    if level > 100:
        warnings.warn(f"volume {level} is above 100, clamping", stacklevel=2)
        level = 100
    return level

print(set_volume(150))      # warning on stderr, then 100 is printed

From here on we mostly **capture** warnings with `catch_warnings(record=True)` and print them, so the output is compact. `record=True` collects `WarningMessage` objects (with `.message`, `.category`, `.filename`, `.lineno`) instead of displaying them.

In [ ]:
def show(caught: list[warnings.WarningMessage]) -> None:
    for w in caught:
        print(f"{w.category.__name__}: {w.message}")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")         # make sure nothing is suppressed inside the block
    set_volume(120)
    set_volume(50)
    set_volume(200)

show(caught)                                 # two warnings

## Categories

Every warning has a category — a subclass of `Warning` (itself a subclass of `Exception`). Filters work on categories, so pick the right one:

| Category | Use for | Shown by default? |
|---|---|---|
| `UserWarning` | the default for `warn()` | yes |
| `DeprecationWarning` | deprecated features, for **developers** using your API | only when triggered by code in `__main__` |
| `PendingDeprecationWarning` | will be deprecated in the future | no |
| `FutureWarning` | deprecated features, for **end users** of an application/notebook (e.g. a behaviour change) | yes |
| `RuntimeWarning` | dubious runtime behaviour (e.g. a coroutine never awaited) | yes |
| `SyntaxWarning` | dubious syntax (e.g. `"\d"` invalid escape) | yes |
| `ResourceWarning` | unclosed files/sockets | no |
| `ImportWarning`, `UnicodeWarning`, `BytesWarning`, `EncodingWarning` | specialised cases | varies |

You can define your own category for your library, so users can filter exactly your warnings:

In [ ]:
class PrecisionWarning(UserWarning):
    """Raised when a conversion may lose precision."""

def to_cents(amount: float) -> int:
    cents = round(amount * 100)
    if abs(cents - amount * 100) > 1e-6:
        warnings.warn(f"{amount} rounded to {cents} cents", PrecisionWarning, stacklevel=2)
    return cents

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(to_cents(12.34), to_cents(9.999))

show(caught)
print(issubclass(PrecisionWarning, Warning), issubclass(Warning, Exception))   # True True

## `stacklevel` — blame the caller

A warning records the file and line where it "happened". With the default `stacklevel=1` that's the `warnings.warn(...)` line **inside your function** — useless to the person calling it. `stacklevel=2` attributes it to the caller's line (the way the C# compiler points at the call site of an `[Obsolete]` member). Use `2` in public functions, more if there are helper layers in between.

In [ ]:
def old_helper_level1() -> None:
    warnings.warn("old_helper is deprecated", DeprecationWarning)                 # stacklevel=1

def old_helper_level2() -> None:
    warnings.warn("old_helper is deprecated", DeprecationWarning, stacklevel=2)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_helper_level1()
    old_helper_level2()   # <- this line is reported

import inspect
def_line = inspect.getsourcelines(old_helper_level1)[1]
for w in caught:
    print(f"reported line {w.lineno} (relative to old_helper_level1's def: {w.lineno - def_line:+d})")

The first warning points into the helper's body (line +1 of its definition); the second points at the calling line further down the cell. Since 3.12 there is also `skip_file_prefixes=` to skip all frames from your package, however deep the call chain.

## Filters: what happens to a warning

Each warning is matched against a list of filters; the first match decides the **action**:

| Action | Effect |
|---|---|
| `"default"` | show once per (module + line) |
| `"once"` | show only the first occurrence, regardless of location |
| `"always"` | show every time |
| `"module"` | show once per module |
| `"ignore"` | never show |
| `"error"` | raise it as an exception |

- `warnings.simplefilter(action, category=Warning)` — match by category only.
- `warnings.filterwarnings(action, message="regex", category=..., module="regex", lineno=0)` — finer matching (`message` is matched against the **start** of the message).

New filters are inserted at the front of the list, so they win. Wrap temporary changes in `catch_warnings()`, which restores the filters on exit — think `#pragma warning disable` / `restore`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", message="volume", category=UserWarning)   # silence one kind
    set_volume(300)                       # ignored
    to_cents(0.001)                       # still reported

show(caught)

### `"error"`: warnings as exceptions

Turning warnings into errors is the `TreatWarningsAsErrors` of Python. It's great in test suites and CI, so deprecations are fixed before the API disappears.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("error", PrecisionWarning)
    try:
        to_cents(1.005)
    except PrecisionWarning as e:
        print(f"{type(e).__name__}: {e}")

print(to_cents(2.50))        # outside the block: back to normal behaviour

⚠️ **Gotcha:** with the `"default"` action a warning is shown only **once per call site** (same message text, category and line): Python remembers it in a per-module `__warningregistry__`. In a loop you'll see one warning, not a hundred — and when testing, a warning that was already shown may seem to "disappear". That's why the examples call `simplefilter("always")` inside `catch_warnings(record=True)`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("default")
    for _ in range(3):
        set_volume(150)                   # same message, same call site, three times

print(len(caught), "warning(s) recorded")  # 1

## Testing that code warns

`catch_warnings(record=True)` is the building block; unit-test frameworks wrap it — `unittest` has `self.assertWarns(Category)` (and pytest has `pytest.warns`) *(see Standard Library › Testing & Debugging › 001 · Unit Testing with unittest)*.

In [ ]:
def assert_warns(category: type[Warning], func, *args) -> None:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        func(*args)
    assert any(issubclass(w.category, category) for w in caught), f"no {category.__name__}"
    print(f"OK: {func.__name__}{args} warned with {category.__name__}")

assert_warns(PrecisionWarning, to_cents, 3.14159)
try:
    assert_warns(PrecisionWarning, to_cents, 3.25)
except AssertionError as e:
    print(f"{type(e).__name__}: {e}")

## `DeprecationWarning` is hidden by default

⚠️ **Gotcha:** the default filters **ignore** `DeprecationWarning` — *unless* the warning is attributed to code in `__main__` (your script, or a notebook cell). So a library calling another library's deprecated API stays silent in production, while your own top-level code gets warned. Another reason to get `stacklevel` right: with `stacklevel=1` the warning is attributed to the library itself and nobody sees it.

In [ ]:
# each filter is (action, message_regex, category, module_regex, lineno)
for action, _msg, category, module, _lineno in warnings.filters:
    if category is DeprecationWarning:
        pattern = getattr(module, "pattern", module)   # a compiled regex, a str, or None
        print(action, category.__name__, pattern or "(any module)")

Let's prove it in a fresh interpreter with a small module in a temp directory. We run the current Python (`sys.executable`) with `subprocess` *(see Standard Library › System & CLI › 004 · Running Processes with subprocess)* and look at its stderr.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

LIB = '''
import warnings
def old_level1():
    warnings.warn("old_level1 is deprecated", DeprecationWarning)
def old_level2():
    warnings.warn("old_level2 is deprecated", DeprecationWarning, stacklevel=2)
'''

def run(*args: str, code: str = "import mylib; mylib.old_level1(); mylib.old_level2()") -> str:
    with tempfile.TemporaryDirectory() as d:
        (Path(d) / "mylib.py").write_text(LIB, encoding="utf-8")
        env = {k: v for k, v in os.environ.items() if k not in ("PYTHONWARNINGS", "PYTHONDEVMODE")}
        env["PYTHON_COLORS"] = "0"                      # plain text tracebacks (3.13+ colours them)
        result = subprocess.run([sys.executable, *args, "-c", code],
                                cwd=d, capture_output=True, text=True, env=env)
        out = result.stderr.strip() or "(no warnings shown)"
        for prefix in {str(Path(d).resolve()), d}:     # hide the machine-specific temp path
            out = out.replace(prefix, "<tmp>")
        return out

print(run())

Only `old_level2` was shown: with `stacklevel=2` the warning is attributed to `<string>` (the `-c` code, i.e. `__main__`), while `old_level1`'s warning is attributed to `mylib` and suppressed.

## Command line: `-W` and `PYTHONWARNINGS`

The same filters can be set from outside the program — useful in CI:

```bash
python -W error script.py                          # all warnings become exceptions
python -W default::DeprecationWarning script.py    # show every deprecation (once per location)
PYTHONWARNINGS=ignore::FutureWarning python app.py
python -X dev script.py                            # "development mode": shows Deprecation/ResourceWarnings and more
```

The format is `action:message:category:module:lineno`, with empty fields matching anything.

In [ ]:
print(run("-W", "default::DeprecationWarning"))          # now both are shown
print("---")
print(run("-W", "error::DeprecationWarning").splitlines()[-1])   # and now it's an exception

## `@warnings.deprecated` (3.13+)

Python 3.13 added a decorator modelled on `[Obsolete]`. At runtime it emits a `DeprecationWarning` (with the right `stacklevel`) whenever the function is called or the class is instantiated, and it stores the message in `__deprecated__`. Static type checkers (mypy, pyright) understand it too and flag uses at "compile time", like CS0618.

```csharp
// C#
[Obsolete("Use CalculateTotal() instead")]
public decimal CalcTotal(IEnumerable<decimal> prices) => CalculateTotal(prices);
```

In [ ]:
@warnings.deprecated("Use calculate_total() instead")
def calc_total(prices: list[float]) -> float:
    return calculate_total(prices)

def calculate_total(prices: list[float]) -> float:
    return round(sum(prices), 2)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(calc_total([1.10, 2.20]))

show(caught)
print(calc_total.__deprecated__)

It works on classes as well (warning on instantiation), and you can pass a different `category=` — e.g. `FutureWarning` for end-user-facing deprecations — or `category=None` to only mark the API for type checkers without a runtime warning.

In [ ]:
@warnings.deprecated("LegacyClient is replaced by Client", category=FutureWarning)
class LegacyClient:
    pass

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LegacyClient()

show(caught)

## Routing warnings into logging

`logging.captureWarnings(True)` redirects all displayed warnings to the logger named **`py.warnings`** (at `WARNING` level), so they end up in the same files/sinks as the rest of your logs *(see Standard Library › Logging & Config › 002 · Configuring Logging)*. Filters still apply — only warnings that would have been shown are logged.

In [ ]:
import logging

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("LOG %(name)s %(levelname)s | %(message)s"))
py_warnings = logging.getLogger("py.warnings")
py_warnings.addHandler(handler)
py_warnings.propagate = False

logging.captureWarnings(True)
with warnings.catch_warnings():
    warnings.simplefilter("always")
    warnings.warn("cache directory is not writable, using memory cache", RuntimeWarning)
logging.captureWarnings(False)

py_warnings.removeHandler(handler)
handler.close()

The logged message is the standard formatted warning, which includes the source file and line (here the cell's temporary file).

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Mark API obsolete | `[Obsolete("msg")]` | `@warnings.deprecated("msg")` (3.13+) or `warnings.warn(msg, DeprecationWarning, stacklevel=2)` |
| Generic developer warning | compiler / analyzer warning | `warnings.warn("msg")` (`UserWarning`) |
| Point at the caller | (automatic) | `stacklevel=2` |
| Suppress locally | `#pragma warning disable` | `with catch_warnings(): simplefilter("ignore", Cat)` |
| Warnings as errors | `TreatWarningsAsErrors` | `simplefilter("error")` / `python -W error` |
| Configure externally | `.editorconfig` / `NoWarn` | `-W` option / `PYTHONWARNINGS` |
| Assert a warning in tests | analyzer tests | `catch_warnings(record=True)` / `assertWarns` |
| Send to logs | — | `logging.captureWarnings(True)` |

## Exercises

1. Write `parse_age(text: str) -> int` that returns the integer age but issues a `UserWarning` (with `stacklevel=2`) when the age is over 130. Capture and print the warning for `parse_age("150")`.
2. Mark a function `old_sum(a, b)` as deprecated with `@warnings.deprecated`, then call it inside a `catch_warnings()` block where `DeprecationWarning` is turned into an error. Catch the exception and print it.
3. Call `parse_age("200")` three times in a loop inside `catch_warnings(record=True)`, first with the `"default"` action and then with `"always"`. How many warnings are recorded each time? (Use a fresh loop for each — and think about why the numbers differ.)

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import warnings

# 1 — warn with stacklevel=2
def parse_age(text: str) -> int:
    age = int(text)
    if age > 130:
        warnings.warn(f"suspicious age {age}", stacklevel=2)
    return age

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    parse_age("150")
print([f"{w.category.__name__}: {w.message}" for w in caught])

# 2 — @deprecated + "error"
@warnings.deprecated("old_sum is deprecated, use sum()")
def old_sum(a, b):
    return a + b

with warnings.catch_warnings():
    warnings.simplefilter("error", DeprecationWarning)
    try:
        old_sum(1, 2)
    except DeprecationWarning as e:
        print(f"{type(e).__name__}: {e}")

# 3 — "default" shows once per call site, "always" every time
for action in ("default", "always"):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter(action)
        for _ in range(3):
            parse_age("200")
    print(action, len(caught))   # default 1, always 3
```

</details>